# Read LSSTCamSources in all bands

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Universite Paris-Saclay
- **Created:** 2026-07-09
- **Last update:** 2026-07-09

## Goal

This notebook is the **offline companion** to `01_FindLSSTCamSourcesInAllbands.ipynb`.
It does **not** touch the Butler: it simply reads back the per-band parquet
files (`objectstats_band_<band>.parquet`) written by notebook 01 into
`OUTPUT_DIR`, reproduces the same summary plots, and adds a new diagnostic
figure: a 2x3 grid of 2D histograms (dispersion `mmag_meas` vs. mean
magnitude) for each band, in the standard LSST order `u, g, r, i, z, y`,
designed to make the high-dispersion tail of the distribution visible in
each band.

**Note:** the parquet files produced at USDF are not copied into this local
directory; point `OUTPUT_DIR` below to wherever you have synced/copied the
`output_objectstats/` folder (or run this notebook directly at USDF).

## Imports

In [ ]:
import glob
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

## Logging

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Le logging est configure et fonctionne dans le notebook !")

## Configuration

**Edit only this cell** to point to the directory containing the
`objectstats_band_<band>.parquet` files written by notebook 01
(`OUTPUT_DIR` there). These constants are only used for plot titles / labels
here (the actual selection was already applied in notebook 01).

In [ ]:
# -- Where notebook 01 wrote its output --------------------------------------
OUTPUT_DIR = "output_objectstats"

# All LSST bands, in the standard display order
BANDS = ["u", "g", "r", "i", "z", "y"]

# -- Magnitude window used in notebook 01 (for plot titles only) -------------
MAG_MIN = 17.0
MAG_MAX = 19.5

# -- Minimum number of visits per band used in notebook 01 (for plot titles) -
MIN_VISITS_PER_BAND = {"u": 20, "g": 50, "r": 50, "i": 50, "z": 50, "y": 20}

log.info(f"Reading per-band object-stats parquet files from '{OUTPUT_DIR}'")

## Read the per-band parquet files

Each file was written by notebook 01 as
`OUTPUT_DIR/objectstats_band_<band>.parquet` and already contains one row per
stable star (object) with the columns produced by `summarize_objects()`:
`object_id, n_visits, ra, dec, flux_mean, flux_std, mag_mean,
sigmaF_over_F_phot, sigmaF_over_F_meas, mmag_meas, mmag_phot, ddf, band`.

In [ ]:
all_band_results = {}

for band in BANDS:
    path = os.path.join(OUTPUT_DIR, f"objectstats_band_{band}.parquet")
    if not os.path.exists(path):
        log.warning(f"Band '{band}': file not found ({path}), skipping")
        continue
    df_band = pd.read_parquet(path)
    all_band_results[band] = df_band
    log.info(f"Band '{band}': {len(df_band)} objects read from {path}")

if not all_band_results:
    raise FileNotFoundError(
        f"No objectstats_band_*.parquet files found in '{OUTPUT_DIR}'. "
        "Check OUTPUT_DIR in the configuration cell above."
    )

## Combine all bands and inspect the results

In [ ]:
df_all = pd.concat(
    [df for df in all_band_results.values() if len(df) > 0],
    ignore_index=True,
)
log.info(f"Total objects across all bands/DDFs: {len(df_all)}")

df_all.head()

In [ ]:
df_all.groupby("band")["mmag_meas"].describe()[["count", "mean", "50%", "std"]]

## Plot: relative photometric scatter (mmag) per band

Boxplot of `mmag_meas` (measured scatter, using `psfFlux`) grouped by band,
in the standard LSST band order `u, g, r, i, z, y`. We expect the largest
scatter in **u** and **y**.

In [ ]:
band_order = [b for b in ["u", "g", "r", "i", "z", "y"] if b in df_all["band"].unique()]

data = [df_all.loc[df_all["band"] == b, "mmag_meas"].dropna().to_numpy() for b in band_order]

fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(data, labels=band_order, showfliers=False)
ax.set_xlabel("Band")
ax.set_ylabel(r"$\sigma_F/F$ (mmag)")
ax.set_title(
    f"Relative PSF-flux scatter, {MAG_MIN:.0f} < mag < {MAG_MAX:.0f}, "
    f">= {MIN_VISITS_PER_BAND} visits/band"
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "mmag_scatter_per_band_reload.png"), dpi=150)
plt.savefig(os.path.join(OUTPUT_DIR, "mmag_scatter_per_band_reload.pdf"))
plt.show()

In [ ]:
# Cross-check: measured scatter vs. photon-noise-only expectation, per band
fig, ax = plt.subplots(figsize=(8, 6))
for b in band_order:
    sub = df_all.loc[df_all["band"] == b]
    ax.scatter(sub["mag_mean"], sub["mmag_meas"], s=4, alpha=0.4, label=b)
ax.set_xlabel("mean magnitude")
ax.set_ylabel(r"$\sigma_F/F$ (mmag), measured")
ax.set_yscale("log")
ax.legend(markerscale=3, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Dispersion vs. magnitude, 2D histogram per band

New figure requested: a `2 x 3` grid of 2D histograms of `mmag_meas`
(dispersion, log-binned) vs. `mag_mean` (magnitude, linear-binned), one panel
per band, in the order `u, g, r, i, z, y`. The dispersion axis is log-scaled
(both the bin edges and the axis) so that the high-dispersion tail -- objects
whose repeated-visit flux scatter is much larger than the bulk of the
population, e.g. blends, variables, or bad cross-matches -- stands out
clearly in every band, rather than being compressed against the x-axis as it
would be on a linear scale.

In [ ]:
def plot_disp_vs_mag(ax, mag, disp, band, mag_min=MAG_MIN, mag_max=MAG_MAX, n_xbins=50, n_ybins=50):
    """2D histogram of dispersion (mmag_meas, log-binned) vs magnitude
    (linear-binned) on a single axis, with a log-scaled y-axis so the
    high-dispersion tail is visible.
    """
    mag = np.asarray(mag, dtype=float)
    disp = np.asarray(disp, dtype=float)
    sel = np.isfinite(mag) & np.isfinite(disp) & (disp > 0)
    mag, disp = mag[sel], disp[sel]

    if len(disp) == 0:
        ax.set_title(f"band {band} (no data)")
        return None

    xedges = np.linspace(mag_min, mag_max, n_xbins + 1)

    # log-spaced bins on the dispersion axis, padded slightly beyond the
    # 0.5th/99.5th percentiles so the tail is not clipped at the bin edge
    ylo = max(np.nanpercentile(disp, 0.5) * 0.8, disp[disp > 0].min())
    yhi = np.nanpercentile(disp, 99.5) * 1.5
    yedges = np.logspace(np.log10(ylo), np.log10(yhi), n_ybins + 1)

    h, xe, ye = np.histogram2d(mag, disp, bins=[xedges, yedges])
    mesh = ax.pcolormesh(xe, ye, h.T, norm=LogNorm(vmin=1, vmax=max(h.max(), 1)), cmap="viridis")
    ax.set_yscale("log")
    ax.set_xlim(mag_min, mag_max)
    ax.set_title(f"band {band}  (N={len(disp)})")
    ax.grid(True, alpha=0.2, which="both")
    return mesh


fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)

band_grid_order = ["u", "g", "r", "i", "z", "y"]

for ax, band in zip(axes.flat, band_grid_order):
    if band not in df_all["band"].unique():
        ax.set_title(f"band {band} (no data)")
        continue
    sub = df_all.loc[df_all["band"] == band]
    mesh = plot_disp_vs_mag(ax, sub["mag_mean"], sub["mmag_meas"], band)
    if mesh is not None:
        fig.colorbar(mesh, ax=ax, label="N objects")

for ax in axes[1, :]:
    ax.set_xlabel("mean magnitude")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\sigma_F/F$ (mmag), measured")

fig.suptitle(
    f"Photometric scatter vs. magnitude per band, {MAG_MIN:.1f} < mag < {MAG_MAX:.1f}",
    y=1.02,
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "mmag_vs_mag_hist2d_per_band.png"), dpi=150, bbox_inches="tight")
plt.savefig(os.path.join(OUTPUT_DIR, "mmag_vs_mag_hist2d_per_band.pdf"), bbox_inches="tight")
plt.show()